<a href="https://colab.research.google.com/github/Rodriamarog/InteligenciaComputacional/blob/main/2_1_preprocessingPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocesamiento PyTorch - California Housing

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Cargar el dataset
data = fetch_california_housing()
X, y = data.data, data.target

print(f"Dataset shape: {X.shape}")
print(f"Feature names: {data.feature_names}")

# Particion 80/10/10
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape[0]} samples")
print(f"Val: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Dataset shape: (20640, 8)
Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train: 16512 samples
Val: 2064 samples
Test: 2064 samples


In [11]:
# Convertir a tensores
X_train_tensor = torch.FloatTensor(X_train)
X_val_tensor = torch.FloatTensor(X_val)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# Indices de features
feature_cols = [0, 1, 2, 3, 4, 5]  # MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup
latlon_cols = [6, 7]  # Latitude, Longitude

# Calcular bounds para outliers usando IQR en train set
features_train = X_train_tensor[:, feature_cols]
Q1 = torch.quantile(features_train, 0.25, dim=0)
Q3 = torch.quantile(features_train, 0.75, dim=0)
IQR = Q3 - Q1
lower_bounds = Q1 - 1.5 * IQR
upper_bounds = Q3 + 1.5 * IQR

print("Bounds para outlier removal calculados")
print(f"Lower bounds: {lower_bounds}")
print(f"Upper bounds: {upper_bounds}")

Bounds para outlier removal calculados
Lower bounds: tensor([  -0.7430,  -10.5000,    2.0386,    0.8657, -616.5000,    1.1520])
Upper bounds: tensor([8.0829e+00, 6.5500e+01, 8.4745e+00, 1.2411e+00, 3.1315e+03, 4.5568e+00])


In [12]:
# Remover outliers en features
features_train_clean = torch.clamp(features_train, lower_bounds, upper_bounds)

# Calcular mean y std para standard scaling
feature_mean = features_train_clean.mean(dim=0)
feature_std = features_train_clean.std(dim=0)

print(f"Feature means: {feature_mean}")
print(f"Feature stds: {feature_std}")

# Funcion para preprocessar datos
def preprocess_data(X_tensor):
    features = X_tensor[:, feature_cols]
    latlon = X_tensor[:, latlon_cols]

   # Quitar outliers
    features_capped = torch.clamp(features, lower_bounds, upper_bounds)

    # Standard scaling
    features_scaled = (features_capped - feature_mean) / feature_std

    # Juntar features procesadas con latitud y lonfitud sin cambios
    X_processed = torch.cat([features_scaled, latlon], dim=1)

    return X_processed

# Aplicar preprocessing
X_train_processed = preprocess_data(X_train_tensor)
X_val_processed = preprocess_data(X_val_tensor)
X_test_processed = preprocess_data(X_test_tensor)

print(f"Datos procesados. Shape: {X_train_processed.shape}")

Feature means: tensor([3.8132e+00, 2.8608e+01, 5.3136e+00, 1.0586e+00, 1.3380e+03, 2.8979e+00])
Feature stds: tensor([1.6681e+00, 1.2602e+01, 1.2467e+00, 8.0810e-02, 7.6491e+02, 6.9065e-01])
Datos procesados. Shape: torch.Size([16512, 8])


In [13]:
# Modelo simple de red neuronal
class SimpleNet(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.fc1 = nn.Linear(8, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Función para entrenar modelo
def train_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = criterion(y_val_pred, y_val)

        train_losses.append(loss.item())
        val_losses.append(val_loss.item())

        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Train Loss = {loss.item():.4f}, Val Loss = {val_loss.item():.4f}")

    return train_losses, val_losses

print("Modelo y funcion de entrenamiento definidos")

Modelo y funcion de entrenamiento definidos


In [14]:
# Modelo 1: Red pequeña
model1 = SimpleNet(hidden_size=32)
print("Entrenando modelo 1 (hidden_size=32)")
train_losses1, val_losses1 = train_model(model1, X_train_processed, y_train_tensor,
                                        X_val_processed, y_val_tensor, epochs=100, lr=0.001)

Entrenando modelo 1 (hidden_size=32)
Epoch 0: Train Loss = 60.5645, Val Loss = 47.3459
Epoch 20: Train Loss = 3.8804, Val Loss = 2.9178
Epoch 40: Train Loss = 1.4328, Val Loss = 1.3655
Epoch 60: Train Loss = 1.3012, Val Loss = 1.3047
Epoch 80: Train Loss = 1.2656, Val Loss = 1.2461


In [15]:
# Modelo 2: Red mas grande
model2 = SimpleNet(hidden_size=128)
print("Entrenando modelo 2 (hidden_size=128)")
train_losses2, val_losses2 = train_model(model2, X_train_processed, y_train_tensor,
                                        X_val_processed, y_val_tensor, epochs=100, lr=0.001)

Entrenando modelo 2 (hidden_size=128)
Epoch 0: Train Loss = 129.8736, Val Loss = 26.0415
Epoch 20: Train Loss = 3.7071, Val Loss = 5.3010
Epoch 40: Train Loss = 1.9629, Val Loss = 1.8441
Epoch 60: Train Loss = 1.1572, Val Loss = 1.1439
Epoch 80: Train Loss = 1.1159, Val Loss = 1.0940


In [16]:
# Modelo 3: Learning rate mas alto
model3 = SimpleNet(hidden_size=64)
print("Entrenando modelo 3 (hidden_size=64, lr=0.01)")
train_losses3, val_losses3 = train_model(model3, X_train_processed, y_train_tensor,
                                        X_val_processed, y_val_tensor, epochs=100, lr=0.01)

Entrenando modelo 3 (hidden_size=64, lr=0.01)
Epoch 0: Train Loss = 4.0344, Val Loss = 193.9962
Epoch 20: Train Loss = 1.4082, Val Loss = 1.9307
Epoch 40: Train Loss = 1.1362, Val Loss = 1.1722
Epoch 60: Train Loss = 0.8095, Val Loss = 0.7966
Epoch 80: Train Loss = 0.6168, Val Loss = 0.6335


In [17]:
# Comparar modelos en validation set
models = [model1, model2, model3]
model_names = ["Modelo 1 (32)", "Modelo 2 (128)", "Modelo 3 (64, lr=0.01)"]
val_losses_final = [val_losses1[-1], val_losses2[-1], val_losses3[-1]]

print("Validation losses finales:")
for name, loss in zip(model_names, val_losses_final):
    print(f"{name}: {loss:.4f}")

# Elegir mejor modelo
best_model_idx = np.argmin(val_losses_final)
best_model = models[best_model_idx]
best_model_name = model_names[best_model_idx]

print(f"\nMejor modelo: {best_model_name}")
print(f"Validation loss: {val_losses_final[best_model_idx]:.4f}")

Validation losses finales:
Modelo 1 (32): 1.2161
Modelo 2 (128): 1.0525
Modelo 3 (64, lr=0.01): 0.5938

Mejor modelo: Modelo 3 (64, lr=0.01)
Validation loss: 0.5938


In [18]:
# Evaluar mejor modelo en test set
best_model.eval()
with torch.no_grad():
    y_test_pred = best_model(X_test_processed)
    test_mse = nn.MSELoss()(y_test_pred, y_test_tensor)
    test_rmse = torch.sqrt(test_mse)

print(f"Evaluación final del {best_model_name}:")
print(f"Test MSE: {test_mse.item():.4f}")
print(f"Test RMSE: {test_rmse.item():.4f}")

# Calcular R cuadrada
y_test_mean = y_test_tensor.mean()
ss_res = torch.sum((y_test_tensor - y_test_pred) ** 2)
ss_tot = torch.sum((y_test_tensor - y_test_mean) ** 2)
r2 = 1 - (ss_res / ss_tot)

print(f"Test R²: {r2.item():.4f}")

Evaluación final del Modelo 3 (64, lr=0.01):
Test MSE: 0.5738
Test RMSE: 0.7575
Test R²: 0.5598
